In [1]:
import torch
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from annoy import AnnoyIndex

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

/home/linuxachin/Desktop/Codes/Two-Tower-Recommendation/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


'cpu'

In [3]:



class TwoTower(nn.Module):
    def __init__(self, product_embed_dim, query_embed_dim, hidden_dim, output_size):
        super().__init__()
        self.product_tower_layer1 = nn.Linear(product_embed_dim, hidden_dim)  # First hidden layer
        self.product_tower_layer2 = nn.Linear(hidden_dim, output_size)

        self.query_tower_layer1 = nn.Linear(query_embed_dim, hidden_dim)     # 3840*512 
        self.query_tower_layer2 = nn.Linear(hidden_dim, output_size)
        self.cos = nn.CosineSimilarity(dim=-1)
        self.sigmoid = nn.Sigmoid()
        self.relu = nn.ReLU()                

    def forward(self, product_embed, query_embed):
        x_p = self.product_tower_layer1(product_embed)  # Apply ReLU after first layer
        x_p = self.relu(x_p)
        x_p = self.product_tower_layer2(x_p)

        x_q = self.query_tower_layer1(query_embed)  # Apply ReLU after second layer
        x_q = self.relu(x_q)
        x_q = self.query_tower_layer2(x_q)
        
        x_p = F.normalize(x_p, p=2, dim=-1)
        x_q = F.normalize(x_q, p=2, dim=-1)
        output = self.sigmoid(self.cos(x_p, x_q))


        return output


In [4]:
files = ['product_brand_embedding.npy', 'product_bullet_point_embedding.npy', 'product_color_embedding.npy', 'product_title_embedding.npy', 'product_description_embedding.npy', 'query_embedding.npy']

data = []
for file in files:
    data.append(np.load(file))

In [5]:
embedding = np.concat((data[0],data[1],data[2],data[3],data[4],data[5]),axis=1)

ANN search

In [15]:
files = ['product_brand_embedding.npy', 'product_bullet_point_embedding.npy', 'product_color_embedding.npy', 'product_title_embedding.npy', 'product_description_embedding.npy']

data = []
for file in files:
    data.append(np.load(file))

In [16]:
embedding = np.concat((data[0],data[1],data[2],data[3],data[4]),axis=1)

In [21]:
query_embedding = np.load('query_embedding.npy')

In [28]:
df = pd.read_csv('data.csv')

In [30]:
df.iloc[2000]

Unnamed: 0                                                         516132
product_id                                                     B07XBLW2GQ
product_title           Mixigoo Makeup Brush Cleaner Dryer - Electric ...
product_description     <b>How long have you not cleaned your makeup b...
product_bullet_point    【Fast Clean & Quick Dry Makeup Brush Clean Mac...
product_brand                                                     mixigoo
product_color                                                       Black
query                                                makeuo brush cleaner
esci_label                                                              E
split                                                               train
binary_label                                                            1
Name: 2000, dtype: object

In [23]:
annoy_index = AnnoyIndex(f=768, metric='euclidean')


In [24]:

for i, embed in enumerate(query_embedding):
    annoy_index.add_item(i, embed)

# Build the index
annoy_index.build(100) 

True

In [31]:
nearest_indices = annoy_index.get_nns_by_vector(query_embedding[2000], 10)

In [32]:
df.iloc[nearest_indices]

,Unnamed: 0,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,query,esci_label,split,binary_label
2000,516132,B07XBLW2GQ,Mixigoo Makeup Brush Cleaner Dryer - Electric ...,<b>How long have you not cleaned your makeup b...,【Fast Clean & Quick Dry Makeup Brush Clean Mac...,mixigoo,Black,makeuo brush cleaner,E,train,1
13856,194840,B083LX58T8,"Makeup Brush Cleaner Dryer, Neeyer Super-Fast ...",Transform Your Makeup Routine With this Electr...,Fast Clean & Dry - With this electric brush cl...,Neeyer,black,makeuo brush cleaner,E,train,1
52353,516136,B0855DXKL8,Senbowe Upgraded Makeup Brush Cleaner and Drye...,<b>Specifications:</b><br> Use power: <3W<br> ...,"【FAST CLEAN QUICK DRY】: Newest design, no need...",senbowe,pink,makeuo brush cleaner,E,train,1
2430,135767,B07Z8S5XDK,"Electric Sonic Vibrating Facial Brush, ZesGood...",<b>ZesGood Cleansing Brush can improve your fa...,【Why Choose ZesGood Face Brush】 Our facial bru...,ZesGood,Green,facial cleansing brush,E,test,1
72707,135770,B08DKX4M9M,"Facial Cleansing Brush,【2021 Upgraded】ETEREAUT...",The Face Brushes for Cleansing and Exfoliating...,🌹THE PERFECT ALL-IN-ONE FACE BRUSH EXFOLIATION...,Etereauty,Gray,facial cleansing brush,E,test,1
3565,492949,B083Z1XGD3,"Facial Cleansing Brush [Newest 2021], PIXNOR W...",facial cleansing brush,A PERFECT ALL-IN-ONE FACIAL CARE SYSTEM. A wat...,PIXNOR,Bean Green,face cleansing brush,E,test,1
97820,492952,B084SBNKWP,EZBASICS Sonic Facial Cleansing Brush made wit...,<p>EZBASICS Facial cleansing Brush- Brings Bac...,Distinctive Facial Brush: Ultra Hygienic Soft ...,EZBASICS,Pink,face cleansing brush,E,test,1
88633,782579,B008VGMWCO,"Oh Yuk Jetted Tub Cleaner for Jacuzzis, Bathtu...",A Cleaner Tub for a Cleaner You! <br/><br/>Oh ...,THE MOST EFFECTIVE JETTED TUB CLEANER - Oh Yuk...,Oh Yuk,Green,jaccuzi tub cleaner brush,E,train,1
90792,172625,B0006B0TJA,Fuller Brush Tub & Shower E-Z Scrubber Brush -...,<p><b>Easily scrubbing tubs and showers</b> wi...,Unique Design: Featuring a specially designed ...,Fuller Brush,Silver,jaccuzi tub cleaner brush,E,train,1
11070,194910,B08664T827,UVC Light Sterilizer Box | Portable Multi-Purp...,<p>Meet our <strong>Portable Home & office UV ...,⚡️ MULTI-PURPOSE UVC STERILIZER – Not only for...,WILLBRITE,White,makeup brush uv sanitizer,E,train,1


Training

In [6]:
labels = pd.read_csv('data.csv')['binary_label'].values

In [7]:
labels

array([1, 0, 1, ..., 1, 0, 1], shape=(100000,))

In [8]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(embedding, labels, test_size=0.2, random_state=42, shuffle=True)

In [9]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train)
y_test = torch.from_numpy(y_test)

In [10]:
train_dataset = TensorDataset(x_train,y_train)
test_dataset = TensorDataset(x_test,y_test)

In [11]:
train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=512, shuffle=False)


In [13]:
model = TwoTower(product_embed_dim=3840,query_embed_dim=768, hidden_dim=512, output_size=32).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

RuntimeError: CUDA error: CUDA-capable device(s) is/are busy or unavailable
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [50]:
from sklearn.metrics import accuracy_score

# Training loop
epochs = 100
for epoch in range(epochs):
    total_correct = 0
    total_samples = 0
    
    for batch_x, batch_y in train_dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device).float()  # Ensure batch_y is float
        
        optimizer.zero_grad()
        outputs = model(batch_x[:, :3840], batch_x[:, 3840:])
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        # Convert probabilities to binary predictions (0 or 1)
        predictions = torch.round(outputs)  # Sigmoid outputs in range [0,1]
        
        # Count correct predictions
        total_correct += (predictions == batch_y).sum().item()
        total_samples += batch_y.size(0)

    accuracy = total_correct / total_samples * 100  # Convert to percentage
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.2f}%')


Epoch [10/100], Loss: 0.4191, Accuracy: 88.94%
Epoch [20/100], Loss: 0.4389, Accuracy: 89.45%
Epoch [30/100], Loss: 0.3985, Accuracy: 89.34%
Epoch [40/100], Loss: 0.4332, Accuracy: 89.60%
Epoch [50/100], Loss: 0.4484, Accuracy: 89.23%
Epoch [60/100], Loss: 0.4251, Accuracy: 89.78%
Epoch [70/100], Loss: 0.4229, Accuracy: 90.16%
Epoch [80/100], Loss: 0.4141, Accuracy: 90.21%
Epoch [90/100], Loss: 0.3923, Accuracy: 90.08%
Epoch [100/100], Loss: 0.3928, Accuracy: 90.00%
